In [ ]:
import pandas as pd
from google.colab import files


# load data
df = pd.read_csv("WhatnotDataRefined1023.csv")

print("Original dataset shape:", df.shape)

# Clean column names
df.columns = df.columns.astype(str).str.strip()
df = df.dropna(axis=1, how="all")

# Remove leftover unnamed columns
df = df.loc[
    :,
    ~df.columns.str.contains(r"^Unnamed", case=False)
]
#rows
df = df.dropna(how="all").copy()

# whitespace
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip()


df = df.replace(r"^\s*$", pd.NA, regex=True)

# numeric bids
for col in ["Starting Bid", "Ending Bid"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# style standardization
df["Style"] = df["Style"].str.upper()

# bid type standardization
df["Bid Type"] = df["Bid Type"].str.strip().str.lower()
df["Bid Type"] = df["Bid Type"].replace({
    "auction": "Auction",
    "sd": "SD"
})

# clothing type standardization
df["Type"] = df["Type"].str.strip().str.lower()

type_mapping = {
    "crew": "Crewneck",
    "crewneck": "Crewneck",
    "crew neck": "Crewneck",

    "quarterzip": "zipup",
    "quarter zip": "zipup",
    "quarter-zip": "zipup",

    "zipup hoodie": "hoodie",
    "zip up hoodie": "hoodie",
    "zip-up hoodie": "hoodie",

    "joggers": "pants",
    "shorts": "pants"
}

df["Type"] = df["Type"].replace(type_mapping)

# zipup standardization
quarterzip_mask = df["Type"].str.contains(
    r"quarter[\s-]*zip", case=False, na=False
)
df.loc[quarterzip_mask, "Type"] = "zipup"

# jacket standardization
is_varsity = df["Type"].str.contains(
    "varsity", case=False, na=False
)

contains_jacket = df["Type"].str.contains(
    "jacket", case=False, na=False
)

contains_detroit = df["Type"].str.contains(
    "detroit", case=False, na=False
)

jacket_mask = is_varsity | contains_jacket | contains_detroit
df.loc[jacket_mask, "Type"] = "Jacket"

# capitalize the first letter of every clothing type
df["Type"] = df["Type"].str.capitalize()

# wear and stain standardization
for col in ["Wear", "Stainage"]:
    df[col] = df[col].str.strip().str.lower()

# size needs to be text
df["Size"] = df["Size"].astype("string").str.strip()

# for review:
print("\nMissing values per column:")
print(df.isna().sum())

print("\nNegative bid records:")
print(df[
    (df["Starting Bid"] < 0) |
    (df["Ending Bid"] < 0)
])

print("\nEnding bids above $100:")
print(df[df["Ending Bid"] > 100])

# Check for exact duplicate rows
print("\nExact duplicate rows:", df.duplicated().sum())

# Display standardized clothing types and counts
print("\nStandardized clothing types:")
print(df["Type"].value_counts(dropna=False))

print(
    "\nTotal entries labeled Jacket:",
    (df["Type"] == "Jacket").sum()
)





print("\nColumn names:")
print(df.columns.tolist())


output_file = "/content/whatnot_data_cleaned23232.csv"
df.to_csv(output_file, index=False)

files.download(output_file)

Original dataset shape: (630, 10)

Missing values per column:
Style           0
Size            0
Starting Bid    0
Ending Bid      0
Type            0
Channel         0
Show Title      0
Bid Type        0
Wear            0
Stainage        0
dtype: int64

Negative bid records:
Empty DataFrame
Columns: [Style, Size, Starting Bid, Ending Bid, Type, Channel, Show Title, Bid Type, Wear, Stainage]
Index: []

Ending bids above $100:
    Style Size  Starting Bid  Ending Bid      Type Channel  \
161    EM    M             5         112    Jacket   Mrtop   
172    EM    M             5         180    Jacket   Mrtop   
186     P    M             5         131    Jacket   Mrtop   
205     P   XL             5         170  Crewneck   Mrtop   
209     P    L             5         160    Jacket   Mrtop   

              Show Title Bid Type      Wear Stainage  
161        Carhartt Show       SD  moderate    clean  
172        Carhartt Show       SD      none    clean  
186  Premium Sports Show       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import altair as alt
alt.data_transformers.disable_max_rows()

dv = df.dropna(subset=["Ending Bid"]).copy()                     # only items that have a final price
dv["stream"] = dv["Channel"] + " | " + dv["Show Title"]     # one livestream
LIGHT = alt.Scale(range=["#f0f4ff", "#7fa6e0"])                  # light color ramp so numbers stay readable

In [ ]:

dv_sold = dv[dv['Ending Bid'] > 0]


base = (alt.Chart(dv_sold)
    .transform_window(item="row_number()", groupby=["stream"]))

raw = (base.mark_circle(size=18, opacity=0.35)
    .encode(x="item:Q",
            y=alt.Y("Ending Bid:Q", title="Ending Bid ($)"),
            color="stream:N",
            tooltip=["stream", "item:Q", "Ending Bid:Q"]))

trend = (base
    .transform_loess("item", "Ending Bid", groupby=["stream", "Bid Type"], bandwidth=0.2)
    .mark_line(strokeWidth=2.5)
    .encode(x="item:Q", y="Ending Bid:Q",
            color="stream:N", strokeDash="Bid Type:N"))

static = (trend).properties(width=620, height=380)

static

alt.Chart(...)

In [ ]:
dv = dv.copy()
dv['item'] = dv.groupby('stream').cumcount() + 1
dv_sold = dv[dv['Ending Bid'] > 0]

streams = sorted(dv['stream'].unique())
YMAX = dv_sold['Ending Bid'].max()

ALL = 'All streams'
SD  = 'All Sudden Death streams'
AUC = 'All Auction streams'

choice = alt.param(
    name='choice', value=ALL,
    bind=alt.binding_select(options=[ALL, SD, AUC] + streams, name='Show: '))

keep = (f"choice == '{ALL}' "
        f"|| (choice == '{SD}'  && datum['Bid Type'] == 'SD') "
        f"|| (choice == '{AUC}' && datum['Bid Type'] == 'Auction') "
        f"|| datum.stream == choice")

base = alt.Chart(dv_sold)

trend = (base
    .add_params(choice)
    .transform_filter(keep)
    .transform_loess("item", "Ending Bid", groupby=["stream", "Bid Type"], bandwidth=0.2)
    .mark_line(strokeWidth=2.5)
    .encode(x=alt.X("item:Q", title="Item order in stream"),
            y=alt.Y("Ending Bid:Q", title="Ending Bid ($)", scale=alt.Scale(domain=[0, YMAX])),
            color=alt.Color("stream:N", scale=alt.Scale(domain=streams)),
            strokeDash="Bid Type:N"))

interactive = trend.properties(width=620, height=380)

interactive

alt.Chart(...)